In [ ]:
import rasterio
import numpy as np

# Input and output paths
input_tif = r"D:\...\worldpop_2020.tif"
output_tif = r"C:\...\worldpop_2020_log.tif"

with rasterio.open(input_tif) as src:
    # 1. Read original data (band 1) and metadata
    arr = src.read(1)
    profile = src.profile
    original_nodata = src.nodata

    # 2. Replace original NoData and negative values with NaN
    arr_clean = np.where(
        (arr == original_nodata) | (arr < 0),
        np.nan,
        arr
    )

    # 3. Apply log10(1 + x) transformation
    log_arr = np.log10(1 + arr_clean)
    # log_arr = np.log10(1 + arr_clean) / np.log10(100)

    # 4. Update output metadata
    profile.update(
        dtype=rasterio.float32,
        count=1,
        nodata=np.nan
    )

    # 5. Write output raster
    with rasterio.open(output_tif, 'w', **profile) as dst:
        dst.write(log_arr.astype(rasterio.float32), 1)

print("log10(1 + x) transformation completed! In output file:")
print(f"- Original NoData and negative values converted to NaN")

In [ ]:
import rasterio
from rasterio.warp import reproject, Resampling
import numpy as np

src_path = r"C:\...\worldpop_2020_log.tif"
dst_path = r"C:\...\NeglectedEvents_251009.tif"
out_path = r"C:\...\worldpop_2020_log_ccy.tif"

# 打开目标文件获取元信息
with rasterio.open(dst_path) as dst:
    dst_profile = dst.profile
    dst_crs = dst.crs
    dst_transform = dst.transform
    dst_height = dst.height
    dst_width = dst.width

# 打开源文件并进行重采样
with rasterio.open(src_path) as src:
    # 创建一个空数组存储重采样结果
    out_arr = np.empty((dst_height, dst_width), dtype=np.float32)
    
    reproject(
        source=rasterio.band(src, 1),
        destination=out_arr,
        src_transform=src.transform,
        src_crs=src.crs,
        dst_transform=dst_transform,
        dst_crs=dst_crs,
        resampling=Resampling.bilinear
    )

    # 更新输出 profile
    dst_profile.update(
        dtype=rasterio.float32,
        count=1,
        nodata=np.nan
    )

    # 写入新文件
    with rasterio.open(out_path, 'w', **dst_profile) as out_ds:
        out_ds.write(out_arr, 1)

print(f"对齐完成！新文件已保存到：{out_path}")

In [ ]:
import rasterio
import numpy as np

def normalize_tiff_01(input_path, output_path):
    try:
        with rasterio.open(input_path) as src:
            # Read data
            pop_data = src.read(1).astype(np.float32)
            nodata_value = src.nodata

            # Convert NoData to NaN
            if nodata_value is not None:
                pop_data = np.where(np.isclose(pop_data, nodata_value), np.nan, pop_data)

            # Get valid data
            valid_data = pop_data[~np.isnan(pop_data)]
            if valid_data.size == 0:
                raise ValueError("No valid pixel data in the file")

            # Calculate min/max values
            min_val = np.nanmin(pop_data)
            max_val = np.nanmax(pop_data)

            # Avoid division by zero
            if max_val == min_val:
                pop_norm = np.zeros_like(pop_data)
            else:
                # 0-1 normalization
                pop_norm = (pop_data - min_val) / (max_val - min_val)

            # Update metadata
            meta = src.meta.copy()
            meta.update(dtype=rasterio.float32, nodata=np.nan)

            # Write output raster
            with rasterio.open(output_path, 'w', **meta) as dst:
                dst.write(pop_norm, 1)

            return True, f"✅ 0-1 normalization completed: {output_path}"

    except Exception as e:
        return False, f"❌ Processing error: {str(e)}"

if __name__ == "__main__":
    input_tif = r"C:\...\worldpop_2020_log_ccy.tif"
    output_tif = r"C:\...\worldpop_2020_log_ccy_norm.tif"

    success, msg = normalize_tiff_01(input_tif, output_tif)
    print(msg)

In [ ]:
import rasterio
import numpy as np

def invert_01_tiff(input_path, output_path):
    """Invert 0-1 normalized raster values (1 - value) while preserving NoData"""
    try:
        with rasterio.open(input_path) as src:
            data = src.read(1).astype(np.float32)
            meta = src.meta.copy()
            nodata = src.nodata if src.nodata is not None else np.nan

            # Convert NoData pixels to NaN
            data = np.where(np.isclose(data, nodata), np.nan, data)

            # Invert values: 1 - original value
            inverted = 1 - data

            # Update output metadata
            meta.update(dtype=np.float32, nodata=np.nan)

            # Write inverted raster to file
            with rasterio.open(output_path, 'w', **meta) as dst:
                dst.write(inverted, 1)

        return True, f"✅ Inversion completed! Output file: {output_path}"

    except Exception as e:
        return False, f"❌ Processing failed: {str(e)}"


if __name__ == "__main__":
    input_tif = r"C:\...\NeglectedEvents_251009.tif"
    output_tif = r"C:\...\NeglectedEvents_251009_inverted.tif"

    success, msg = invert_01_tiff(input_tif, output_tif)
    print(msg)

In [ ]:
import rasterio
import numpy as np

def multiply_without_resampling(base_tif, coeff_tif, output_tif):
    """Multiply two aligned rasters pixel-wise without resampling"""
    with rasterio.open(base_tif) as base_src, \
         rasterio.open(coeff_tif) as coeff_src:

        # Read event raster data
        base_data = base_src.read(1).astype(np.float32)
        base_nodata = base_src.nodata if base_src.nodata is not None else np.nan

        # Read population raster data
        coeff_data = coeff_src.read(1).astype(np.float32)
        coeff_nodata = coeff_src.nodata if coeff_src.nodata is not None else np.nan
        
        # Set 0 values in coefficient raster to NaN
        coeff_data[coeff_data == 0] = np.nan
        
        # Initialize result array with NaN
        result = np.full_like(base_data, np.nan, dtype=np.float32)

        # Iterate over each pixel of base raster
        for i in range(base_src.height):
            for j in range(base_src.width):
                # Skip NoData pixels in base raster
                if np.isclose(base_data[i, j], base_nodata):
                    continue

                # Get geographic coordinates of current pixel
                x, y = base_src.xy(i, j)

                # Get corresponding row and column in coefficient raster
                row, col = coeff_src.index(x, y)

                # Check if indices are within bounds
                if 0 <= row < coeff_src.height and 0 <= col < coeff_src.width:
                    cval = coeff_data[row, col]
                    # Multiply if coefficient value is valid
                    if not np.isclose(cval, coeff_nodata) and not np.isnan(cval):
                        result[i, j] = base_data[i, j] * cval

        # Save output raster
        meta = base_src.meta.copy()
        meta.update(dtype=np.float32, nodata=np.nan)
        with rasterio.open(output_tif, 'w', **meta) as dst:
            dst.write(result, 1)

    return True, f"✅ Completed! Output: {output_tif}"

if __name__ == "__main__":
    base_tif = r"C:\...\NeglectedEvents_251009_inverted.tif"
    coeff_tif = r"C:\...\worldpop_2020_log_ccy_norm.tif"
    output_tif = r"C:\...\NeglectedEvents_251009_PendInverted.tif"

    success, msg = multiply_without_resampling(base_tif, coeff_tif, output_tif)
    print(msg)

In [ ]:
import rasterio
import numpy as np

def invert_normalize_tiff(input_path, output_path):
    """Inverse normalize raster values to 0-1 range while preserving NoData"""
    try:
        with rasterio.open(input_path) as src:
            # Read data and metadata
            data = src.read(1).astype(np.float32)
            meta = src.meta.copy()
            nodata = src.nodata if src.nodata is not None else np.nan

            # Calculate valid data range (exclude NoData)
            valid_data = data[~np.isnan(data)]
            if valid_data.size == 0:
                raise ValueError("No valid pixels in input file, cannot perform inverse normalization")
            
            min_orig = np.min(valid_data)
            max_orig = np.max(valid_data)

            # Avoid division by zero
            if max_orig == min_orig:
                inverted_data = 1.0 - data
            else:
                # Core inverse normalization formula
                inverted_data = (max_orig - data) / (max_orig - min_orig)

            # Restore NoData regions
            inverted_data[np.isnan(data)] = np.nan

            # Update metadata
            meta.update(
                dtype=np.float32,
                nodata=np.nan
            )

            # Write output raster
            with rasterio.open(output_path, 'w', **meta) as dst:
                dst.write(inverted_data, 1)

        return True, f"✅ Inverse normalization completed! Output file: {output_path}" \
                     f"\nOriginal range: [{min_orig:.4f}, {max_orig:.4f}]" \
                     f"\nNormalized range: [{np.min(inverted_data[~np.isnan(inverted_data)]):.4f}, {np.max(inverted_data[~np.isnan(inverted_data)]):.4f}]"

    except Exception as e:
        return False, f"❌ Processing error: {str(e)}"

if __name__ == "__main__":
    # Input and output paths
    input_tif = r"C:\...\NeglectedEvents_251009_PendInverted.tif"
    output_tif = r"C:\...\NeglectedEvents_251009_Pop.tif"

    # Run inverse normalization
    success, msg = invert_normalize_tiff(input_tif, output_tif)
    print(msg)